# 00 - Does everything work?  *(~1 minute)*

> **Presenter script.** "Before we train anything, let's make sure the machine
> under this notebook can actually do the job. This is the boring cell that
> saves you twenty minutes of confusion later."

By the end of this notebook we will have:

1. confirmed Python, PyTorch, NumPy and matplotlib are installed,
2. proved that training works **on a plain CPU** - no graphics card anywhere,
3. built every dataset the rest of the stream needs, into `data/`.

Nothing here takes longer than a minute.

In [ ]:
# --- boilerplate: make `import minigpt` work no matter where Jupyter started ---
import pathlib
import sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "minigpt").is_dir())
sys.path.insert(0, str(ROOT))

import torch

torch.set_num_threads(4)  # plenty for a model this small; more threads is not faster
print("repo root:", ROOT)

## 1. What is installed?

**PyTorch** is the library that does the maths: it holds the model's numbers and
works out which direction to nudge them. Everything else is scaffolding.

In [ ]:
import platform

import matplotlib
import numpy as np

print(f"python      : {platform.python_version()}  ({platform.system()})")
print(f"torch       : {torch.__version__}")
print(f"numpy       : {np.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")
print(f"CPU threads : {torch.get_num_threads()}")
print(f"CUDA GPU    : {torch.cuda.is_available()}  <- False is completely fine, that is the point")

## 2. Can this CPU actually train something?

Let's train the smallest possible "model": a single number that has to learn to
be `7`. If the loss falls, the machinery works.

> **loss** - one number that says how wrong the model currently is. Lower is
> better. Training is nothing more than "make the loss go down".

In [ ]:
import time

torch.manual_seed(0)

guess = torch.zeros(1, requires_grad=True)          # the model: one adjustable number
target = torch.tensor([7.0])                        # the right answer
optimizer = torch.optim.SGD([guess], lr=0.2)        # the thing that does the nudging

started = time.time()
for step in range(21):
    loss = (guess - target) ** 2                    # how wrong are we?
    optimizer.zero_grad()
    loss.backward()                                 # which way should we nudge?
    optimizer.step()                                # nudge
    if step % 5 == 0:
        print(f"step {step:2d} | guess {guess.item():6.3f} | loss {loss.item():8.4f}")

print(f"\ntook {time.time() - started:.2f}s on the CPU - and that is the entire idea of training.")

## 3. A quick speed check

Our real model is about **160,000 numbers** instead of one. Let's time a single
training step so we know what to expect during the live runs.

In [ ]:
from minigpt.model import DEFAULT_BATCH_SIZE, MiniGPT, default_config

model = MiniGPT(default_config(vocab_size=80))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randint(0, 80, (DEFAULT_BATCH_SIZE, model.cfg.block_size))
y = torch.randint(0, 80, (DEFAULT_BATCH_SIZE, model.cfg.block_size))

for _ in range(3):  # warm-up, so we time steady-state speed
    _, loss = model(x, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

started = time.time()
for _ in range(10):
    _, loss = model(x, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
ms = (time.time() - started) / 10 * 1000

print(f"model size      : {model.num_params():,} parameters")
print(f"one step        : {ms:.0f} ms")
print(f"1200 steps      : about {ms * 1200 / 1000:.0f} seconds")
print()
print("If '1200 steps' is much more than ~3 minutes, set USE_PREBAKED = True in the")
print("later notebooks and load the results that were baked in advance.")

## 4. Build the datasets

This writes the training corpus, the held-out validation set, the "new domain"
corpus for notebook 04, and the question/answer pairs for notebook 05 - all into
`data/`.

There is **no big download**: the corpus is generated from a seeded template
grammar, so it is byte-for-byte identical on every machine and works with the
network unplugged. Notebook 01 pulls this apart step by step.

In [ ]:
from minigpt import data
from minigpt.paths import ARTIFACTS, CHECKPOINTS, DATA, ensure_dirs

ensure_dirs()
summary = data.prepare_all()

In [ ]:
print("files in data/:")
for path in sorted(DATA.glob("*")):
    print(f"  {path.name:24} {path.stat().st_size / 1024:8.1f} KB")

## 5. Optional: is the network available?

Nothing in this repo needs the internet. But if you *do* have a connection you
can also train on the public-domain "tiny shakespeare" corpus
(`data.load_base_corpus("shakespeare")`). This cell just reports whether that
option is open - it never fails the notebook.

In [ ]:
text = data.fetch_tiny_shakespeare(timeout=5)
if text:
    print(f"network OK - tiny shakespeare is available ({len(text):,} characters)")
else:
    print("no network (or the download failed) - that is fine, we use the offline corpus")

## 6. Are the pre-baked results here?

`scripts/pretrain_all.py` bakes every checkpoint and loss curve in advance so a
live stream never has to wait. Check what is already on disk.

In [ ]:
ckpts = sorted(p.name for p in CHECKPOINTS.glob("*.pt"))
arts = sorted(p.name for p in ARTIFACTS.glob("*.json"))
print("checkpoints/:", ckpts or "(none yet)")
print("artifacts/  :", arts or "(none yet)")
if not ckpts:
    print("\nRun this once, before you go live:\n  python scripts/pretrain_all.py")

## Checklist

- [x] PyTorch imports and trains on the CPU
- [x] `data/` is populated
- [x] you know roughly how long a training run will take on *this* machine

**Next:** `01_data.ipynb` - where the text comes from, and why cleaning it is
most of the job.